# Mushroom Classification

## Problem Definition
Classifying mushrooms as edible or poisonous based on their physical characteristics (odor, gill size, bruising, etc.).

- Learning type: Supervised Learning
- Problem type: Binary Classification
- Algorithm: Categorical Naive Bayes
- Target: class (e = edible, p = poisonous)

In [1]:
import kagglehub

path = kagglehub.dataset_download("uciml/mushroom-classification")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Abdel\.cache\kagglehub\datasets\uciml\mushroom-classification\versions\1


In [2]:
import os
import shutil

RAW_DATA_DIR = "../data/raw"
os.makedirs(RAW_DATA_DIR, exist_ok=True)

source_file = os.path.join(path, "mushrooms.csv")
destination_file = os.path.join(RAW_DATA_DIR, "mushrooms.csv")

shutil.copy(source_file, destination_file)

print("Data saved to:", destination_file)

Data saved to: ../data/raw\mushrooms.csv


In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/mushrooms.csv")

print("Shape:", df.shape)
df.head()

Shape: (8124, 23)


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   class                     8124 non-null   str  
 1   cap-shape                 8124 non-null   str  
 2   cap-surface               8124 non-null   str  
 3   cap-color                 8124 non-null   str  
 4   bruises                   8124 non-null   str  
 5   odor                      8124 non-null   str  
 6   gill-attachment           8124 non-null   str  
 7   gill-spacing              8124 non-null   str  
 8   gill-size                 8124 non-null   str  
 9   gill-color                8124 non-null   str  
 10  stalk-shape               8124 non-null   str  
 11  stalk-root                8124 non-null   str  
 12  stalk-surface-above-ring  8124 non-null   str  
 13  stalk-surface-below-ring  8124 non-null   str  
 14  stalk-color-above-ring    8124 non-null   str  
 15

In [5]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values -> {sorted(df[col].unique())}")

class: 2 unique values -> ['e', 'p']
cap-shape: 6 unique values -> ['b', 'c', 'f', 'k', 's', 'x']
cap-surface: 4 unique values -> ['f', 'g', 's', 'y']
cap-color: 10 unique values -> ['b', 'c', 'e', 'g', 'n', 'p', 'r', 'u', 'w', 'y']
bruises: 2 unique values -> ['f', 't']
odor: 9 unique values -> ['a', 'c', 'f', 'l', 'm', 'n', 'p', 's', 'y']
gill-attachment: 2 unique values -> ['a', 'f']
gill-spacing: 2 unique values -> ['c', 'w']
gill-size: 2 unique values -> ['b', 'n']
gill-color: 12 unique values -> ['b', 'e', 'g', 'h', 'k', 'n', 'o', 'p', 'r', 'u', 'w', 'y']
stalk-shape: 2 unique values -> ['e', 't']
stalk-root: 5 unique values -> ['?', 'b', 'c', 'e', 'r']
stalk-surface-above-ring: 4 unique values -> ['f', 'k', 's', 'y']
stalk-surface-below-ring: 4 unique values -> ['f', 'k', 's', 'y']
stalk-color-above-ring: 9 unique values -> ['b', 'c', 'e', 'g', 'n', 'o', 'p', 'w', 'y']
stalk-color-below-ring: 9 unique values -> ['b', 'c', 'e', 'g', 'n', 'o', 'p', 'w', 'y']
veil-type: 1 unique va

In [6]:
print(df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("\n'?' count per column:")
print((df == "?").sum()[(df == "?").sum() > 0])

0
Duplicate rows: 0

'?' count per column:
stalk-root    2480
dtype: int64


In [7]:
print(df["class"].value_counts())
print(df["class"].value_counts(normalize=True) * 100)

class
e    4208
p    3916
Name: count, dtype: int64
class
e    51.797144
p    48.202856
Name: proportion, dtype: float64


In [8]:
df = df.drop(columns=["veil-type"])
df["stalk-root"] = df["stalk-root"].replace("?", "Unknown")

print(df.shape)
print(df["stalk-root"].value_counts())

(8124, 22)
stalk-root
b          3776
Unknown    2480
e          1120
c           556
r           192
Name: count, dtype: int64


In [9]:
print((pd.crosstab(df["odor"], df["class"], normalize="index") * 100).round(1))

class      e      p
odor               
a      100.0    0.0
c        0.0  100.0
f        0.0  100.0
l      100.0    0.0
m        0.0  100.0
n       96.6    3.4
p        0.0  100.0
s        0.0  100.0
y        0.0  100.0


In [10]:
for col in ["bruises", "gill-size", "stalk-root"]:
    print(f"--- {col} ---")
    print((pd.crosstab(df[col], df["class"], normalize="index") * 100).round(1))
    print()

--- bruises ---
class       e     p
bruises            
f        30.7  69.3
t        81.5  18.5

--- gill-size ---
class         e     p
gill-size            
b          69.9  30.1
n          11.5  88.5

--- stalk-root ---
class           e     p
stalk-root             
Unknown      29.0  71.0
b            50.8  49.2
c            92.1   7.9
e            77.1  22.9
r           100.0   0.0



In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (6499, 21) | Test: (1625, 21)


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

def make_model(alpha=1.0):
    return Pipeline([
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ("model", CategoricalNB(alpha=alpha)),
    ])

In [13]:
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline = make_model(alpha=1.0)
res = cross_validate(baseline, X_train, y_train, cv=cv,
                      scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
                      params={"model__sample_weight": None} if False else None)

# ملحوظة: precision/recall/f1 هنا pos_label الافتراضي هيبقى "p" أبجديًا بعد الترتيب
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label="p"),
    "recall": make_scorer(recall_score, pos_label="p"),
    "f1": make_scorer(f1_score, pos_label="p"),
    "roc_auc": make_scorer(lambda yt, proba: proba, needs_proba=True) if False else "roc_auc",
}

res = cross_validate(baseline, X_train, y_train, cv=cv, scoring=scoring)

for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    print(f"{metric}: {res[f'test_{metric}'].mean():.4f} ± {res[f'test_{metric}'].std():.4f}")

c:\Users\Abdel\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py:978: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\Abdel\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_scorer.py", line 174, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "c:\Users\Abdel\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_scorer.py", line 469, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
                        ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Abdel\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\Abdel\AppData\Local\Prog

accuracy: 0.9494 ± 0.0109
precision: 0.9933 ± 0.0024
recall: 0.9011 ± 0.0215
f1: 0.9448 ± 0.0124
roc_auc: 0.9975 ± 0.0006


In [14]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label="p"),
    "recall": make_scorer(recall_score, pos_label="p"),
    "f1": make_scorer(f1_score, pos_label="p"),
    "roc_auc": "roc_auc",
}

baseline = make_model(alpha=1.0)
res = cross_validate(baseline, X_train, y_train, cv=cv, scoring=scoring)

for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    print(f"{metric}: {res[f'test_{metric}'].mean():.4f} ± {res[f'test_{metric}'].std():.4f}")

accuracy: 0.9494 ± 0.0109
precision: 0.9933 ± 0.0024
recall: 0.9011 ± 0.0215
f1: 0.9448 ± 0.0124
roc_auc: 0.9975 ± 0.0006


In [15]:
from sklearn.model_selection import GridSearchCV

param_grid = {"model__alpha": [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]}

grid = GridSearchCV(make_model(), param_grid, cv=cv, scoring="recall_macro", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best alpha:", grid.best_params_)
print("Best CV recall_macro:", grid.best_score_)

Best alpha: {'model__alpha': 0.001}
Best CV recall_macro: 0.9928476069721525


In [16]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

best_model = grid.best_estimator_
pred = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)[:, list(best_model.named_steps["model"].classes_).index("p")]

print("Accuracy :", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, pos_label="p"))
print("Recall   :", recall_score(y_test, pred, pos_label="p"))
print("F1       :", f1_score(y_test, pred, pos_label="p"))
print("ROC-AUC  :", roc_auc_score(y_test, proba))
print("\nConfusion Matrix (rows=true, cols=pred, order=[e, p]):")
print(confusion_matrix(y_test, pred, labels=["e", "p"]))

Accuracy : 0.9907692307692307
Precision: 0.9910485933503836
Recall   : 0.9897828863346104
F1       : 0.9904153354632588
ROC-AUC  : 0.9995570966166428

Confusion Matrix (rows=true, cols=pred, order=[e, p]):
[[835   7]
 [  8 775]]


In [17]:
import numpy as np

yt = y_test.values
rng = np.random.RandomState(42)
rows = []
for _ in range(1000):
    idx = rng.randint(0, len(yt), len(yt))
    if len(np.unique(yt[idx])) < 2:
        continue
    rows.append([
        accuracy_score(yt[idx], pred[idx]),
        precision_score(yt[idx], pred[idx], pos_label="p", zero_division=0),
        recall_score(yt[idx], pred[idx], pos_label="p", zero_division=0),
        roc_auc_score(yt[idx], proba[idx]),
    ])

ci = np.percentile(rows, [2.5, 97.5], axis=0)
for name, lo, hi in zip(["Accuracy", "Precision", "Recall", "ROC-AUC"], ci[0], ci[1]):
    print(f"{name}: 95% CI [{lo:.4f}, {hi:.4f}]")

Accuracy: 95% CI [0.9858, 0.9957]
Precision: 95% CI [0.9839, 0.9974]
Recall: 95% CI [0.9819, 0.9962]
ROC-AUC: 95% CI [0.9990, 1.0000]


In [18]:
test_df = X_test.copy()
test_df["y_true"] = y_test.values
test_df["y_pred"] = pred
test_df["proba"] = proba

fn = test_df[(test_df["y_true"] == "p") & (test_df["y_pred"] == "e")]
fp = test_df[(test_df["y_true"] == "e") & (test_df["y_pred"] == "p")]

print("False Negatives (poisonous -> edible):", len(fn))
print(fn[["odor", "bruises", "gill-size", "stalk-root"]])

print("\nFalse Positives (edible -> poisonous):", len(fp))
print(fp[["odor", "bruises", "gill-size", "stalk-root"]])

False Negatives (poisonous -> edible): 8
     odor bruises gill-size stalk-root
5927    n       t         b          b
4364    n       t         n          b
5423    n       t         b          b
5717    n       t         n          b
5237    n       t         n          b
5617    n       t         b          b
4882    n       t         b          b
5369    n       t         b          b

False Positives (edible -> poisonous): 7
     odor bruises gill-size stalk-root
5122    n       f         n          b
5318    n       f         n          b
5582    n       f         n          b
5630    n       f         n          b
5794    n       f         n          b
5117    n       f         n          b
5546    n       f         n          b


In [19]:
import sys
sys.path.append("..")
from preprocessing import MushroomPreprocessor

In [20]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/mushroom_categorical_nb_pipeline.pkl")
print("Saved.")

reloaded = joblib.load("../models/mushroom_categorical_nb_pipeline.pkl")
print("Round trip OK:", np.allclose(reloaded.predict_proba(X_test)[:, 1], proba) or
      np.allclose(reloaded.predict_proba(X_test)[:, 0], proba))

Saved.
Round trip OK: True
